<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


<h1>实验：用于 XOR 的简单神经网络</h1>


<h2>目标</h2><p>完成本实验后，你将能够：</p> 
<ul><li> 创建一个具有多个神经元的神经网络模型来对简单函数建模。</li></ul>


<h2>目录</h2>
<p>在本实验中，你将了解使用一个隐藏层神经网络对带噪声的 XOR 数据进行分类需要多少个神经元。</p>


- [神经网络模块与训练函数](#Neural-Network-Module-and-Training-Function)
- [生成数据](#Make-Some-Data)
- [一个神经元](#One-Neuron)
- [两个神经元](#Two-Neurons)
- [三个神经元](#Three-Neurons)

<p>预计所需时间：<strong>25 分钟</strong></p>
<hr>


<h2>准备工作</h2>


我们需要以下库


In [ ]:
!pip3 install torch torchvision torchaudio
!pip install matplotlib

In [ ]:
# 导入本实验所需的库
# 允许我们使用数组来操作和存储数据
import numpy as np
# PyTorch 库
import torch
# PyTorch 神经网络
import torch.nn as nn
# 允许我们使用激活函数
import torch.nn.functional as F
# 用于绘制数据和损失曲线
import matplotlib.pyplot as plt 
from matplotlib.colors import ListedColormap
# 用于帮助创建数据集并执行小批量训练
from torch.utils.data import Dataset, DataLoader

使用以下函数绘制数据：


In [ ]:
# Plot the data

def plot_decision_regions_2class(model,data_set):
    cmap_light = ListedColormap(['#FFAAAA', '#AAFFAA', '#00AAFF'])
    cmap_bold = ListedColormap(['#FF0000', '#00FF00', '#00AAFF'])
    X = data_set.x.numpy()
    y = data_set.y.numpy()
    h = .02
    x_min, x_max = X[:, 0].min() - 0.1 , X[:, 0].max() + 0.1 
    y_min, y_max = X[:, 1].min() - 0.1 , X[:, 1].max() + 0.1 
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),np.arange(y_min, y_max, h))
    XX = torch.Tensor(np.c_[xx.ravel(), yy.ravel()])

    yhat = np.logical_not((model(XX)[:, 0] > 0.5).numpy()).reshape(xx.shape)
    plt.pcolormesh(xx, yy, yhat, cmap=cmap_light, shading='auto')
    plt.plot(X[y[:, 0] == 0, 0], X[y[:, 0] == 0, 1], 'o', label='y=0')
    plt.plot(X[y[:, 0] == 1, 0], X[y[:, 0] == 1, 1], 'ro', label='y=1')
    plt.title("decision region")
    plt.legend()

使用以下函数计算准确率：


In [ ]:
# Calculate the accuracy

def accuracy(model, data_set):
    # Rounds prediction to nearest integer 0 or 1
    # Checks if prediction matches the actual values and returns accuracy rate
    return np.mean(data_set.y.view(-1).numpy() == (model(data_set.x)[:, 0] > 0.5).numpy())

<!--用于分隔主题的空格-->


<h2 id="Model">神经网络模块与训练函数</h2> 


定义神经网络模块或类：


In [ ]:
# Define the class Net with one hidden layer 

class Net(nn.Module):
    
    # Constructor
    def __init__(self, D_in, H, D_out):
        super(Net, self).__init__()
        # D_in is the input size of the first layer (size of input layer)
        # H is the outpout size of the first layer and the input size of the second layer (size of hidden layer)
        # D_out is the output size of the second layer (size of output layer)
        self.linear1 = nn.Linear(D_in, H)
        self.linear2 = nn.Linear(H, D_out)

    # Prediction    
    def forward(self, x):
        # Puts x through first layer then sigmoid function
        x = torch.sigmoid(self.linear1(x)) 
        # Puts result of previous line through second layer then sigmoid function
        x = torch.sigmoid(self.linear2(x))
        # Output is a number between 0 and 1 due to the sigmoid function. Whichever the output is closer to, 0 or 1, is the class prediction
        return x

定义训练模型的函数：


In [ ]:
# Function to Train the Model

def train(data_set, model, criterion, train_loader, optimizer, epochs=5):
    # Lists to keep track of cost and accuracy
    COST = []
    ACC = []
    # Number of times we train on the entire dataset
    for epoch in range(epochs):
        # Total loss over epoch
        total=0
        # For batch in train laoder
        for x, y in train_loader:
            # Resets the calculated gradient value, this must be done each time as it accumulates if we do not reset
            optimizer.zero_grad()
            # Makes a prediction based on X value
            yhat = model(x)
            # Measures the loss between prediction and acutal Y value
            loss = criterion(yhat, y)
            # Calculates the gradient value with respect to each weight and bias
            loss.backward()
            # Updates the weight and bias according to calculated gradient value
            optimizer.step()
            # Cumulates loss 
            total+=loss.item()
        # Saves cost and accuracy
        ACC.append(accuracy(model, data_set))
        COST.append(total)
        
    # Prints Cost vs Epoch graph
    fig, ax1 = plt.subplots()
    color = 'tab:red'
    ax1.plot(COST, color=color)
    ax1.set_xlabel('epoch', color=color)
    ax1.set_ylabel('total loss', color=color)
    ax1.tick_params(axis='y', color=color)
    
    # Prints Accuracy vs Epoch graph
    ax2 = ax1.twinx()  
    color = 'tab:blue'
    ax2.set_ylabel('accuracy', color=color)  # we already handled the x-label with ax1
    ax2.plot(ACC, color=color)
    ax2.tick_params(axis='y', color=color)
    fig.tight_layout()  # otherwise the right y-label is slightly clipped
    
    plt.show()

    return COST

<!--用于分隔主题的空格-->


<h2 id="Makeup_Data">生成数据</h2> 


数据集类：


In [ ]:
# Define the class XOR_Data

class XOR_Data(Dataset):
    
    # Constructor
    # N_s is the size of the dataset
    def __init__(self, N_s=100):
        # Create a N_s by 2 array for the X values representing the coordinates
        self.x = torch.zeros((N_s, 2))
        # Create a N_s by 1 array for the class the X value belongs to
        self.y = torch.zeros((N_s, 1))
        # Split the dataset into 4 sections
        for i in range(N_s // 4):
            # Create data centered around (0,0) of class 0
            self.x[i, :] = torch.Tensor([0.0, 0.0]) 
            self.y[i, 0] = torch.Tensor([0.0])

            # Create data centered around (0,1) of class 1
            self.x[i + N_s // 4, :] = torch.Tensor([0.0, 1.0])
            self.y[i + N_s // 4, 0] = torch.Tensor([1.0])
    
            # Create data centered around (1,0) of class 1
            self.x[i + N_s // 2, :] = torch.Tensor([1.0, 0.0])
            self.y[i + N_s // 2, 0] = torch.Tensor([1.0])
    
            # Create data centered around (1,1) of class 0
            self.x[i + 3 * N_s // 4, :] = torch.Tensor([1.0, 1.0])
            self.y[i + 3 * N_s // 4, 0] = torch.Tensor([0.0])

            # Add some noise to the X values to make them different
            self.x = self.x + 0.01 * torch.randn((N_s, 2))
        self.len = N_s

    # Getter
    def __getitem__(self, index):    
        return self.x[index],self.y[index]
    
    # Get Length
    def __len__(self):
        return self.len
    
    # Plot the data
    def plot_stuff(self):
        plt.plot(self.x[self.y[:, 0] == 0, 0].numpy(), self.x[self.y[:, 0] == 0, 1].numpy(), 'o', label="y=0")
        plt.plot(self.x[self.y[:, 0] == 1, 0].numpy(), self.x[self.y[:, 0] == 1, 1].numpy(), 'ro', label="y=1")
        plt.legend()

数据集对象：


In [ ]:
# Create dataset object

data_set = XOR_Data()
data_set.plot_stuff()

<!--用于分隔主题的空格-->


<h2 id="One">一个神经元</h2> 


<h3>试一试</h3>


创建一个在隐藏层中具有一个神经元的神经网络 <code>model</code>。然后使用以下代码训练它：


In [ ]:
# Practice: create a model with one neuron
# Type your code here
model = Net(2, 1, 1)

双击<b>此处</b>查看解决方案。

<!-- 
model = Net(2, 1, 1)
-->


In [ ]:
# Train the model

learning_rate = 0.1
# We create a criterion which will measure loss
criterion = nn.BCELoss()
# Create an optimizer that updates model parameters using the learning rate and gradient
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
# Create a Data Loader for the training data with a batch size of 1 
train_loader = DataLoader(dataset=data_set, batch_size=1)
# Using the training function train the model on 500 epochs
LOSS12 = train(data_set, model, criterion, train_loader, optimizer, epochs=500)
# Plot the data with decision boundaries
plot_decision_regions_2class(model, data_set)

<!--用于分隔主题的空格-->


<h2 id="Two">两个神经元</h2> 


<h3>试一试</h3>


创建一个在隐藏层中具有两个神经元的神经网络 <code>model</code>。然后使用以下代码训练它：


In [ ]:
# Practice: create a model with two neuron
# Type your code here
model = Net(2, 2, 1)

双击<b>此处</b>查看解决方案。

<!-- 
model = Net(2, 2, 1)
-->


In [ ]:
# Train the model

learning_rate = 0.1
# We create a criterion which will measure loss
criterion = nn.BCELoss()
# Create an optimizer with the model parameters and learning rate
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
# Create a Data Loader for the training data with a batch size of 1 
train_loader = DataLoader(dataset=data_set, batch_size=1)
# Using the training function train the model on 500 epochs
LOSS12 = train(data_set, model, criterion, train_loader, optimizer, epochs=500)
# Plot the data with decision boundaries
plot_decision_regions_2class(model, data_set)

<!--用于分隔主题的空格-->


<h2 id="Three">三个神经元</h2> 


<h3>试一试</h3>


创建一个在隐藏层中具有三个神经元的神经网络 <code>model</code>。然后使用以下代码训练它：


In [ ]:
# Practice: create a model with two neuron
# Type your code here
model = Net(2, 3, 1)

双击<b>此处</b>查看解决方案。

<!-- 
model = Net(2, 3, 1)
-->


In [ ]:
# Train the model

learning_rate = 0.1
# We create a criterion which will measure loss
criterion = nn.BCELoss()
# Create an optimizer with the model parameters and learning rate
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
# Create a Data Loader for the training data with a batch size of 1 
train_loader = DataLoader(dataset=data_set, batch_size=1)
# Using the training function train the model on 500 epochs
LOSS12 = train(data_set, model, criterion, train_loader, optimizer, epochs=500)
# Plot the data with decision boundaries
plot_decision_regions_2class(model, data_set)

## tensorflow 测试

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.activations import linear, relu, sigmoid
import matplotlib.pyplot as plt

import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

In [ ]:
tf.random.set_seed(12)
model = Sequential(
    [               
        ### START CODE HERE ### 
        tf.keras.Input(shape=(2,)),     # @REPLACE 
        Dense(2, activation='relu', name = "L1"), # @REPLACE 
        Dense(1, activation='sigmoid', name = "L2"),  # @REPLACE  使用 sigmoid 输出概率，与二分类 BinaryCrossentropy 匹配
        ### END CODE HERE ### 
    ], name = "my_model" 
)
model.summary()


In [ ]:
print(f"data_set x:{data_set.x[50:55, 0]}")
print(f"data_set x:{data_set.x[50:55, 1]}")
print(f"data_set y:{data_set.y[50:55, 0]}")

In [ ]:
model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(),  # 二分类交叉熵损失
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    metrics=['accuracy']  # 同时监控准确率
)
# 训练模型，20% 数据用于验证
# 注意：必须设置 shuffle=True，否则 TensorFlow 会按顺序取最后 20% 作为验证集，
# 而 XOR_Data 的后 25 个样本全是类别 0，导致验证集不具代表性，验证准确率虚假达到 1.0
history = model.fit(
    data_set.x.numpy(), data_set.y.numpy(),
    epochs=200,
    validation_split=0.2,
    shuffle=True,  # 每轮训练前打乱数据，同时验证集也会从打乱后的数据中随机抽取
    verbose=0  # 不输出每轮训练日志，减少输出量
)


绘制训练过程：损失与准确率曲线


In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

# 设置支持中文的字体，按常见 Linux/Windows 中文字体顺序尝试
chinese_fonts = ['SimHei', 'WenQuanYi Micro Hei', 'Noto Sans CJK SC', 'Noto Sans CJK JP', 'Microsoft YaHei', 'DejaVu Sans']
available_fonts = {f.name for f in mpl.font_manager.fontManager.ttflist}
selected_font = next((f for f in chinese_fonts if f in available_fonts), 'DejaVu Sans')
plt.rcParams['font.sans-serif'] = [selected_font]
plt.rcParams['axes.unicode_minus'] = False  # 正确显示负号

# 绘制训练与验证损失曲线
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='训练损失')
plt.plot(history.history['val_loss'], label='验证损失')
plt.xlabel('轮次')
plt.ylabel('损失')
plt.title('训练与验证损失')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='训练准确率')
plt.plot(history.history['val_accuracy'], label='验证准确率')
plt.xlabel('轮次')
plt.ylabel('准确率')
plt.title('训练与验证准确率')
plt.legend()

plt.tight_layout()
plt.show()

# 输出最终验证结果
print(f"最终训练准确率: {history.history['accuracy'][-1]:.4f}")
print(f"最终验证准确率: {history.history['val_accuracy'][-1]:.4f}")
print(f"最终训练损失: {history.history['loss'][-1]:.4f}")
print(f"最终验证损失: {history.history['val_loss'][-1]:.4f}")


<!--用于分隔主题的空格-->


<h2>关于作者：</h2> 

<a href="https://www.linkedin.com/in/joseph-s-50398b136/">Joseph Santarcangelo</a> 拥有电气工程博士学位，他的研究重点是利用机器学习、信号处理和计算机视觉来确定视频如何影响人类认知。Joseph 自获得博士学位以来一直在 IBM 工作。


其他贡献者：<a href="https://www.linkedin.com/in/michelleccarey/">Michelle Carey</a>、<a href="https://www.linkedin.com/in/jiahui-mavis-zhou-a4537814a">Mavis Zhou</a>



<!--## Change Log

|  Date (YYYY-MM-DD) |  Version | Changed By  |  Change Description |
|---|---|---|---|
| 2025-07-11  | 2.0  | Sathya  |  Converted to Jupyterlab current |
| 2020-09-23  | 2.0  | Shubham  |  Migrated Lab to Markdown and added to course repo in GitLab |-->



<hr>



## <h3 align="center"> © IBM Corporation. All rights reserved. <h3/>
